# ERA5+SIA → STConvS2S 3D Dataset
## Trabalho 3 — Tema 4 | CIC1205 Machine Learning

Gera `ERA5+SIA_XLV.nc` a partir do `ERA5+SIA.nc` original.


---
## 1. Imports e Configuração

In [2]:
import gc
import os
import numpy as np
import xarray as xr
import netCDF4 as nc

INPUT_PATH  = "ERA5+SIA.nc"
OUTPUT_PATH = "ERA5+SIA_XLV.nc"

# Reduza para 200 se houver MemoryError
CHUNK_SIZE = 500

print(f"Input : {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"netCDF4: {nc.__version__}")

Input : ERA5+SIA.nc
Output: ERA5+SIA_XLV.nc
netCDF4: 1.7.2


---
## 2. Constantes e Mapeamento de Canais

In [ ]:
N_LAT, N_LON = 9, 11

# Níveis: [1000=superfície, 700=troposfera média, 200=troposfera alta]
LEVELS = np.array([1000.0, 700.0, 200.0])

# 7 canais no formato 3D (espelha CHANNEL_NAMES do notebook de referência + speed, w)
CHANNEL_NAMES_3D = ['tp', 'r', 't', 'u', 'v', 'speed', 'w']

N_LEVELS   = len(LEVELS)            # 3
N_CHANNELS = len(CHANNEL_NAMES_3D)  # 7

print(f"Níveis   : {LEVELS}  (idx 0=superfície)")
print(f"Canais 3D: {CHANNEL_NAMES_3D}")
print(f"Shape por timestep: ({N_LAT}, {N_LON}, {N_LEVELS}, {N_CHANNELS})")

Níveis   : [1000.  700.  200.]  (idx 0=superfície)
Canais 3D: ['tp', 'r', 't', 'u', 'v', 'speed', 'w']
Shape por timestep: (9, 11, 3, 7)


---
## 3. `build_volume_3d` 

```python
# Notebook de referência (ERA5 bruto):
volume[:, :, 0, 0] = tp_all[i]   # tp só no nível 0
volume[:, :, :, 1] = t_all[i]    # t em todos os níveis
...

# Este notebook (ERA5+SIA flat→3D):
volume[:, :, 0, 0] = flat[:, :, 0]   # tp só no nível 0
volume[:, :, 0, 1] = flat[:, :, 3]   # r1000 → nível 0
volume[:, :, 1, 1] = flat[:, :, 2]   # r700  → nível 1
...
```


In [ ]:
def build_volume_3d(flat: np.ndarray) -> np.ndarray:
    """
    Converte um timestep flat (H, W, 19) em volume 3D (H, W, L, C).
    Espelha build_timestep_cache_3d do stconvs2s-era5-data-3d.ipynb.

    Convenção (igual ao notebook de referência):
      - tp APENAS no nivel 0 (1000 hPa), zeros em 700 e 200
      - demais variaveis presentes em todos os 3 niveis
    """
    volume = np.zeros((N_LAT, N_LON, N_LEVELS, N_CHANNELS), dtype=np.float32)

    # canal 0: tp — apenas no nivel 0 (1000 hPa = superficie)
    volume[:, :, 0, 0] = flat[:, :, 0]

    # canal 1: r (umidade relativa)
    volume[:, :, 0, 1] = flat[:, :, 3]   # r1000
    volume[:, :, 1, 1] = flat[:, :, 2]   # r700
    volume[:, :, 2, 1] = flat[:, :, 1]   # r200

    # canal 2: t (temperatura)
    volume[:, :, 0, 2] = flat[:, :, 6]   # t1000
    volume[:, :, 1, 2] = flat[:, :, 5]   # t700
    volume[:, :, 2, 2] = flat[:, :, 4]   # t200

    # canal 3: u (componente zonal)
    volume[:, :, 0, 3] = flat[:, :, 9]   # u1000
    volume[:, :, 1, 3] = flat[:, :, 8]   # u700
    volume[:, :, 2, 3] = flat[:, :, 7]   # u200

    # canal 4: v (componente meridional)
    volume[:, :, 0, 4] = flat[:, :, 12]  # v1000
    volume[:, :, 1, 4] = flat[:, :, 11]  # v700
    volume[:, :, 2, 4] = flat[:, :, 10]  # v200

    # canal 5: speed 
    volume[:, :, 0, 5] = flat[:, :, 15]  # speed1000
    volume[:, :, 1, 5] = flat[:, :, 14]  # speed700
    volume[:, :, 2, 5] = flat[:, :, 13]  # speed200

    # canal 6: w (velocidade vertical)
    volume[:, :, 0, 6] = flat[:, :, 18]  # w1000
    volume[:, :, 1, 6] = flat[:, :, 17]  # w700
    volume[:, :, 2, 6] = flat[:, :, 16]  # w200

    return volume


def build_sample_3d(flat_sample: np.ndarray) -> np.ndarray:
    """
    Aplica build_volume_3d a cada timestep de uma amostra.
    Entrada : (T, H, W, 19)
    Saida   : (T, H, W, L, C)
    """
    T = flat_sample.shape[0]
    out = np.zeros((T, N_LAT, N_LON, N_LEVELS, N_CHANNELS), dtype=np.float32)
    for t in range(T):
        out[t] = build_volume_3d(flat_sample[t])
    return out


print("build_volume_3d e build_sample_3d definidas.")

build_volume_3d e build_sample_3d definidas.


---
## 4. Validação com Lote Pequeno

In [ ]:
print(f"Abrindo {INPUT_PATH} ...")
ds = xr.open_dataset(INPUT_PATH)
print(ds)

print("\nCarregando 4 amostras para validacao...")
x_mini = ds["x"].values[:4].astype(np.float32)
T_win  = x_mini.shape[1]

x_mini_3d = np.stack([build_sample_3d(x_mini[i]) for i in range(4)], axis=0)
print(f"  x_mini shape   : {x_mini.shape}")
print(f"  x_mini_3d shape: {x_mini_3d.shape}")


assert np.all(x_mini_3d[:, :, :, :, 1:, 0] == 0),           "ERRO: tp deve ser zero em niveis 1 e 2"
assert np.allclose(x_mini[:, :, :, :, 0], x_mini_3d[:, :, :, :, 0, 0]), "ERRO: tp nao bate"
assert np.allclose(x_mini[:, :, :, :, 3], x_mini_3d[:, :, :, :, 0, 1]), "ERRO: r1000 nao bate"
assert np.allclose(x_mini[:, :, :, :, 1], x_mini_3d[:, :, :, :, 2, 1]), "ERRO: r200 nao bate"
assert np.allclose(x_mini[:, :, :, :, 18], x_mini_3d[:, :, :, :, 0, 6]), "ERRO: w1000 nao bate"
assert np.allclose(x_mini[:, :, :, :, 16], x_mini_3d[:, :, :, :, 2, 6]), "ERRO: w200 nao bate"
assert not np.isnan(x_mini_3d).any(), "ERRO: NaNs encontrados"

del x_mini, x_mini_3d
print("Validacao OK.")

Abrindo ERA5+SIA.nc ...
<xarray.Dataset>
Dimensions:    (sample: 89482, timestep: 5, lat: 9, lon: 11, channel: 19)
Coordinates:
  * sample     (sample) int64 0 1 2 3 4 5 ... 89477 89478 89479 89480 89481
  * timestep   (timestep) int64 0 1 2 3 4
    timestamp  (sample) datetime64[ns] ...
  * lat        (lat) float64 -21.8 -22.05 -22.3 -22.55 ... -23.3 -23.55 -23.8
  * lon        (lon) float64 -45.05 -44.8 -44.55 -44.3 ... -43.05 -42.8 -42.55
  * channel    (channel) int64 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18
Data variables:
    x          (sample, timestep, lat, lon, channel) float64 ...
    y          (sample, timestep, lat, lon, channel) float64 ...

Carregando 4 amostras para validacao...
  x_mini shape   : (4, 5, 9, 11, 19)
  x_mini_3d shape: (4, 5, 9, 11, 3, 7)
Validacao OK.


---
## 5. Carregamento de `x` e `y`



In [ ]:
print("Carregando x...")
x_np = ds["x"].values.astype(np.float32)   # (N, T, H, W, 19)
N, T, _, _, _ = x_np.shape
print(f"  shape: {x_np.shape}")

print("Carregando y (todos os 19 canais)...")
y_np = ds["y"].values.astype(np.float32)   # (N, T, H, W, 19)
print(f"  shape: {y_np.shape}")

Carregando x...
  shape: (89482, 5, 9, 11, 19)
Carregando y (todos os 19 canais)...
  shape: (89482, 5, 9, 11, 19)


---
## 6. Salvando `ERA5+SIA_XLV_3D.nc`

`x` e `y` salvos com as **mesmas 6 dimensões**: `(N, T, H, W, L, C)`.
Usa `netCDF4` direto para evitar o `MemoryError` do xarray.


In [ ]:
def save_xlv_3d(ds_original, x_flat, y_flat, output_path, chunk_size=CHUNK_SIZE):
    """
    Converte x_flat e y_flat (N, T, H, W, 19) para 3D e salva.
    Escreve em chunks de chunk_size amostras por vez.
    x e y ficam com as mesmas 6 dimensoes: (N, T, H, W, L, C).
    """
    N, T, H, W, _ = x_flat.shape
    print(f"Salvando {N} amostras em chunks de {chunk_size}...")

    with nc.Dataset(output_path, 'w', format='NETCDF4') as f:
        f.source             = 'ERA5+SIA'
        f.representation     = 'XLV_3D'
        f.channel_names_info = str(CHANNEL_NAMES_3D)
        f.level_units        = 'hPa'
        f.history            = 'Gerado a partir de ERA5+SIA.nc. Adaptado de stconvs2s-era5-data-3d.ipynb.'

        f.createDimension('sample',   N)
        f.createDimension('timestep', T)
        f.createDimension('lat',      H)
        f.createDimension('lon',      W)
        f.createDimension('level',    N_LEVELS)
        f.createDimension('channel',  N_CHANNELS)

        sv  = f.createVariable('sample',   'i8', ('sample',));   sv[:]  = ds_original.sample.values
        tv  = f.createVariable('timestep', 'i8', ('timestep',)); tv[:]  = ds_original.timestep.values
        lav = f.createVariable('lat',      'f4', ('lat',));      lav[:] = ds_original.lat.values;  lav.units = 'degrees_north'
        lov = f.createVariable('lon',      'f4', ('lon',));      lov[:] = ds_original.lon.values;  lov.units = 'degrees_east'
        lev = f.createVariable('level',    'f4', ('level',));    lev[:] = LEVELS;                  lev.units = 'hPa'

        if 'timestamp' in ds_original:
            tsv = f.createVariable('timestamp', 'i8', ('sample',))
            tsv[:] = ds_original['timestamp'].values.astype('int64')

        dims6 = ('sample', 'timestep', 'lat', 'lon', 'level', 'channel')
        cs6   = (min(chunk_size, N), T, H, W, N_LEVELS, N_CHANNELS)

        xv = f.createVariable('x', 'f4', dims6, chunksizes=cs6)
        xv.description   = 'Input 3D. channel_names: ' + str(CHANNEL_NAMES_3D)
        xv.tp_convention = 'tp nonzero only at level_idx=0 (1000 hPa)'

        # y com as mesmas 6 dimensoes do x — necessario para o dataset.py
        yv = f.createVariable('y', 'f4', dims6, chunksizes=cs6)
        yv.description   = 'Target 3D — mesmo formato de x. Usar target_levels=0, target_channels=0 para tp.'

        for i in range(0, N, chunk_size):
            end = min(i + chunk_size, N)
            cs  = end - i
            xv[i:end] = np.stack([build_sample_3d(x_flat[i:end][k]) for k in range(cs)], axis=0)
            yv[i:end] = np.stack([build_sample_3d(y_flat[i:end][k]) for k in range(cs)], axis=0)
            print(f"  [{end:>6}/{N}]...", end='\r')

    gb = os.path.getsize(output_path) / 1e9
    print(f"\n  Salvo! ({gb:.2f} GB)")


save_xlv_3d(ds, x_np, y_np, OUTPUT_PATH)

del x_np, y_np
gc.collect()
print("Memoria liberada.")

Salvando 89482 amostras em chunks de 500...
  [ 89482/89482]...
  Salvo! (7.44 GB)
Memoria liberada.


---
## 7. Checklist de Validação

Espelha o checklist do notebook de referência.


In [ ]:
print(f"Carregando {OUTPUT_PATH} para validacao...")
loaded = xr.load_dataset(OUTPUT_PATH)
print(loaded)
print(f"\nx shape: {loaded.x.shape}")
print(f"y shape: {loaded.y.shape}")

tp_surface     = loaded.x.values[:, :, :, :, 0, 0]
tp_upper       = loaded.x.values[:, :, :, :, 1:, 0]
tp_max         = float(tp_surface.max())
print(f"\ntp max: {tp_max:.2f} mm/h")

checks = {
    "x.shape == y.shape":
        loaded.x.shape == loaded.y.shape,

    "y tem 6 dimensoes (igual ao x)":
        loaded.y.ndim == 6,

    "sem NaNs em x/y":
        not (np.isnan(loaded.x.values).any() or np.isnan(loaded.y.values).any()),

    "nivel order [1000, 700, 200]":
        list(loaded.level.values) == list(LEVELS),

    "canal 0 e tp":
        loaded.attrs.get('channel_names_info', '') == str(CHANNEL_NAMES_3D)
        and CHANNEL_NAMES_3D[0] == 'tp',

    "tp zero nos niveis superiores":
        np.all(tp_upper == 0),

    "tp nao-nulo na superficie (level 0)":
        np.any(tp_surface != 0),

    "7 canais":
        loaded.sizes["channel"] == N_CHANNELS,

    "3 niveis":
        loaded.sizes["level"] == N_LEVELS,

    "window length 5":
        loaded.sizes["timestep"] == 5,

    "tp em mm/h — max > 1":
        tp_max > 1.0,

    "tp em mm/h — max plausivel < 5000":
        tp_max < 5000.0,
}

print("\nChecklist:")
all_ok = True
for nome, ok in checks.items():
    print(f"  [{'OK' if ok else 'ERRO'}] {nome}")
    all_ok = all_ok and ok

assert all_ok, "Checklist falhou!"
print(f"\nTodos os checks passaram.")
print(f"  {OUTPUT_PATH}: {os.path.getsize(OUTPUT_PATH)/1e9:.2f} GB")

Carregando ERA5+SIA_XLV_3D.nc para validacao...
<xarray.Dataset>
Dimensions:    (sample: 89482, timestep: 5, lat: 9, lon: 11, level: 3,
                channel: 7)
Coordinates:
  * sample     (sample) int64 0 1 2 3 4 5 ... 89477 89478 89479 89480 89481
  * timestep   (timestep) int64 0 1 2 3 4
  * lat        (lat) float32 -21.8 -22.05 -22.3 -22.55 ... -23.3 -23.55 -23.8
  * lon        (lon) float32 -45.05 -44.8 -44.55 -44.3 ... -43.05 -42.8 -42.55
  * level      (level) float32 1e+03 700.0 200.0
Dimensions without coordinates: channel
Data variables:
    timestamp  (sample) int64 1293854400000000000 ... 1727722800000000000
    x          (sample, timestep, lat, lon, level, channel) float32 0.3154 .....
    y          (sample, timestep, lat, lon, level, channel) float32 0.8698 .....
Attributes:
    source:              ERA5+SIA
    representation:      XLV_3D
    channel_names_info:  ['tp', 'r', 't', 'u', 'v', 'speed', 'w']
    level_units:         hPa
    history:             Gerado a 

---
## Teste para comparar dados

In [4]:
ds = xr.open_dataset("ERA5+SIA.nc")
print(ds)
print(ds.sizes)
print(ds.x.sizes)
print(ds.y.sizes)

<xarray.Dataset>
Dimensions:    (sample: 89482, timestep: 5, lat: 9, lon: 11, channel: 19)
Coordinates:
  * sample     (sample) int64 0 1 2 3 4 5 ... 89477 89478 89479 89480 89481
  * timestep   (timestep) int64 0 1 2 3 4
    timestamp  (sample) datetime64[ns] ...
  * lat        (lat) float64 -21.8 -22.05 -22.3 -22.55 ... -23.3 -23.55 -23.8
  * lon        (lon) float64 -45.05 -44.8 -44.55 -44.3 ... -43.05 -42.8 -42.55
  * channel    (channel) int64 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18
Data variables:
    x          (sample, timestep, lat, lon, channel) float64 ...
    y          (sample, timestep, lat, lon, channel) float64 ...
Frozen({'sample': 89482, 'timestep': 5, 'lat': 9, 'lon': 11, 'channel': 19})
Frozen({'sample': 89482, 'timestep': 5, 'lat': 9, 'lon': 11, 'channel': 19})
Frozen({'sample': 89482, 'timestep': 5, 'lat': 9, 'lon': 11, 'channel': 19})


---
## 8. Resumo

| Etapa | O que fez |
|-------|-----------|
| Seção 2 | Definiu constantes e mapeamento dos 19 canais (fonte: `WebsirenesTarget.py`) |
| Seção 3 | Implementou `build_volume_3d` |
| Seção 4 | Validou conversão com 4 amostras |
| Seção 5 | Carregou `x` e `y` completos (19 canais cada) |
| Seção 6 | Converteu e salvou em chunks via `netCDF4` — `x` e `y` com 6D iguais |
| Seção 7 | Checklist de validação |


### Rodar o baseline após gerar o arquivo

```bash
python main.py --cuda 1 -i 1 -v 4 -m stconvs2s-r -e 1 -p 100 --plot --dropout 0.5 -dsp "ERA5+SIA.nc" --target-channels 0 --target-levels 0 -r "production_run" -w 0      

```
### Rodar o Level-Aware após gerar o arquivo

```bash
python main.py --cuda 1 -i 1 -v 4 -m stconvs2s-r -e 1 -p 100 --plot --dropout 0.5 -dsp "ERA5+SIA_XLV.nc" --target-channels 0 --target-levels 0 -r "production_run" -w 0     

```
